In [2]:
import requests
import re

url = "https://gasolinamexico.com.mx/estados/sonora/hermosillo/"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-MX,es;q=0.9",
}

response = requests.get(url, headers=headers, timeout=15)
html = response.text

print(f"Status: {response.status_code}")
print(f"Tamaño HTML: {len(html)} chars")
print()

# ── Busca todas las etiquetas <script> ──────────────────────────
scripts = re.findall(r'<script[^>]*>(.*?)</script>', html, re.DOTALL)

print(f"Scripts encontrados: {len(scripts)}")
print()

# ── Busca cuál script tiene coordenadas (números con decimales largos) ──
for i, script in enumerate(scripts):
    if re.search(r'-?\d{2,3}\.\d{4,}', script):  # lat/lng pattern
        print(f"{'='*60}")
        print(f"SCRIPT #{i} (tiene coordenadas):")
        print(f"{'='*60}")
        print(script[:2000])  # Primeros 2000 chars
        print()

Status: 200
Tamaño HTML: 69715 chars

Scripts encontrados: 15

SCRIPT #12 (tiene coordenadas):


	  var stations = [
		["ACEITES Y COMBUSTIBLES J.R. S.A. DE C.V.", 29.17184, -110.9152], ["aga combustibles sa de cv", 29.06998, -110.977], ["AGROCOMBUSTIBLES EL CHAMIZAL S.A. DE C.V.", 29.06755, -111.0908], ["ALMA DELIA MILLAN LOPEZ", 29.1071, -111.0235], ["ARMANDO EMMANUEL VILLA DOMINGUEZ", 29.04831, -110.984], ["ARMANDO VILLA ENCINAS", 29.00823, -110.944], ["ARMANDO VILLA ENCINAS", 29.14818, -111.0064], ["AUTO SERVICIO BACHOCO SA DE CV", 29.11785, -110.9516], ["auto servicio lori sa de cv", 29.01382, -110.9227], ["AUTO SERVICIO SACRAMENTO SA DE CV", 29.1437, -111.0087], ["AUTOSERVICIO LA CANDELARIA SA DE CV ", 29.14538, -110.9741], ["AUTOSERVICIO LA CANDELARIA SA DE CV ", 29.13135, -111.0231], ["AUTOSERVICIO PASEO DEL CANAL, S.A. DE C.V.", 29.0701, -110.9771], ["AUTOSERVICIO ROSALES, S.A. DE C.V.", 29.06892, -110.9779], ["AUTOSERVICIO TRANSVERSAL, S.A. DE C.V.", 29.08838, -110.9665], ["A

In [4]:
import re
import json

start         = html.find("var stations = [")
bracket_start = html.find("[", start)
bracket_end   = html.find("];", bracket_start)
raw           = html[bracket_start : bracket_end + 1]

# ── Fix: elimina coma(s) + whitespace antes del ] final ──────
raw_clean = re.sub(r',\s*\]$', ']', raw.strip())

data = json.loads(raw_clean)
print(f"✅ Estaciones encontradas: {len(data)}")
for item in data[:3]:
    print(f"  Nombre  : {item[0]}")
    print(f"  Latitud : {item[1]}")
    print(f"  Longitud: {item[2]}")

✅ Estaciones encontradas: 156
  Nombre  : ACEITES Y COMBUSTIBLES J.R. S.A. DE C.V.
  Latitud : 29.17184
  Longitud: -110.9152
  Nombre  : aga combustibles sa de cv
  Latitud : 29.06998
  Longitud: -110.977
  Nombre  : AGROCOMBUSTIBLES EL CHAMIZAL S.A. DE C.V.
  Latitud : 29.06755
  Longitud: -111.0908


In [5]:
# sina/scraping/gas_locations.py
import re
import json
import time
import logging
import requests
import unicodedata

from pathlib import Path
from bs4 import BeautifulSoup
from sina.config.paths import GAS_DATA

log = logging.getLogger(__name__)

BASE_URL = "https://gasolinamexico.com.mx/estados"

# ============================================================
#  HELPERS
# ============================================================
def _slugify(text: str) -> str:
    """
    Convierte 'Baja California Sur' → 'baja-california-sur'
    para armar la URL correctamente.
    """
    text = text.lower().strip()
    # Quita acentos
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    # Reemplaza espacios y caracteres especiales por guión
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = text.strip("-")
    return text


def _extract_stations_from_html(html: str) -> list | None:

    start = html.find("var stations = [")
    if start == -1:
        log.warning("No se encontró 'var stations' en el HTML")
        return None

    bracket_start = html.find("[", start)
    bracket_end   = html.find("];", bracket_start)

    if bracket_end == -1:
        log.warning("No se encontró el cierre ]; del array")
        return None

    raw       = html[bracket_start : bracket_end + 1]
    raw_clean = re.sub(r',\s*\]$', ']', raw.strip())  # ← fix trailing comma

    try:
        return json.loads(raw_clean)
    except json.JSONDecodeError as e:
        log.warning(f"JSONDecodeError: {e}")
        log.warning(f"Raw final: {repr(raw_clean[-100:])}")
        return None


# ============================================================
#  CORE
# ============================================================
def scrape_municipio_locations(
    estado: str,
    municipio: str,
    delay: float = 1.0,
) -> list[dict] | None:
    """
    Scrapea gasolinamexico.com.mx para un municipio específico.
    Retorna lista de dicts con {nombre, latitud, longitud} o None si falla.
    """
    estado_slug    = _slugify(estado)
    municipio_slug = _slugify(municipio)
    url = f"{BASE_URL}/{estado_slug}/{municipio_slug}/"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "es-MX,es;q=0.9",
        "Referer": "https://gasolinamexico.com.mx/",
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)

        if response.status_code == 404:
            log.warning(f"  ⚠️  Página no encontrada: {url}")
            return None

        response.raise_for_status()

    except requests.RequestException as e:
        log.error(f"  ❌ Error de red para {estado}/{municipio}: {e}")
        return None

    stations_raw = _extract_stations_from_html(response.text)

    if not stations_raw:
        log.warning(f"  ⚠️  No se encontró array de estaciones en {url}")
        return None

    # Normaliza a lista de dicts
    # El array es: ["Nombre", lat, lng]
    stations = []
    for item in stations_raw:
        try:
            stations.append({
                "nombre"  : item[0],
                "latitud" : float(item[1]),
                "longitud": float(item[2]),
                "estado"  : estado,
                "municipio": municipio,
            })
        except (IndexError, ValueError, TypeError) as e:
            log.warning(f"    Item malformado {item}: {e}")
            continue

    log.info(f"  ✅ {municipio}: {len(stations)} ubicaciones encontradas")
    time.sleep(delay)

    return stations

In [6]:
resultado = scrape_municipio_locations("sonora", "hermosillo")

In [7]:
resultado

[{'nombre': 'ACEITES Y COMBUSTIBLES J.R. S.A. DE C.V.',
  'latitud': 29.17184,
  'longitud': -110.9152,
  'estado': 'sonora',
  'municipio': 'hermosillo'},
 {'nombre': 'aga combustibles sa de cv',
  'latitud': 29.06998,
  'longitud': -110.977,
  'estado': 'sonora',
  'municipio': 'hermosillo'},
 {'nombre': 'AGROCOMBUSTIBLES EL CHAMIZAL S.A. DE C.V.',
  'latitud': 29.06755,
  'longitud': -111.0908,
  'estado': 'sonora',
  'municipio': 'hermosillo'},
 {'nombre': 'ALMA DELIA MILLAN LOPEZ',
  'latitud': 29.1071,
  'longitud': -111.0235,
  'estado': 'sonora',
  'municipio': 'hermosillo'},
 {'nombre': 'ARMANDO EMMANUEL VILLA DOMINGUEZ',
  'latitud': 29.04831,
  'longitud': -110.984,
  'estado': 'sonora',
  'municipio': 'hermosillo'},
 {'nombre': 'ARMANDO VILLA ENCINAS',
  'latitud': 29.00823,
  'longitud': -110.944,
  'estado': 'sonora',
  'municipio': 'hermosillo'},
 {'nombre': 'ARMANDO VILLA ENCINAS',
  'latitud': 29.14818,
  'longitud': -111.0064,
  'estado': 'sonora',
  'municipio': 'her

In [ ]:
# En tu notebook, trae datos de Hermosillo y mira el campo Numero
from sina.scraping.gas import df_gas_prices

df = df_gas_prices("sonora", "hermosillo")
print(df[["Numero", "Nombre", "Direccion"]].head(10).to_string())

🗄️  Usando SQLite local: sina_data.db
Status  : 200
Size    : 111.7 KB
Content : application/json; charset=utf-8
Keys : ['Success', 'Errors', 'Value']
Columnas disponibles: ['Numero', 'Direccion', 'Producto', 'SubProducto', 'PrecioVigente', 'EntidadFederativaId', 'MunicipioId', 'Nombre']
  Premium    ← Premium (con un índice de octano ([RON+MON]/2) mínimo de 91)
  Magna      ← Regular (con un índice de octano ([RON+MON]/2) mínimo de 87)
  Diesel     ← Diésel Automotriz [contenido mayor de azufre a 15 mg/kg y contenido máximo de azufre de 500 mg/kg]
  Diesel     ← Diésel
  Premium    ← Premium (con contenido mínimo de 92 octanos)
  Magna      ← Regular (con contenido menor a 92 octanos)
  Diesel     ← Diésel de Ultra Bajo Azufre (DUBA) [contenido máximo de azufre de 15 mg/kg]
                 Numero                                       Nombre                                                     Direccion
0  PL/10156/EXP/ES/2015                         FEDERICO LÓPEZ LÓPEZ               

In [8]:
import requests
from bs4 import BeautifulSoup

url = "https://gasolinamexico.com.mx/estados/sonora/hermosillo/"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers, timeout=15)
soup     = BeautifulSoup(response.text, "html.parser")

# ── Busca todos los links a páginas de estación ──────────────
links = [
    a["href"] 
    for a in soup.find_all("a", href=True)
    if "/estacion/" in a["href"]
]

print(f"Links encontrados: {len(links)}")
print("\nPrimeros 5:")
for l in links[:5]:
    print(f"  {l}")

Links encontrados: 162

Primeros 5:
  https://gasolinamexico.com.mx/estacion/13392/aceites-y-combustibles-jr-sa-de-cv/
  https://gasolinamexico.com.mx/estacion/14320/aga-combustibles-sa-de-cv/
  https://gasolinamexico.com.mx/estacion/5230/agrocombustibles-el-chamizal-sa-de-cv/
  https://gasolinamexico.com.mx/estacion/4677/alma-delia-millan-lopez/
  https://gasolinamexico.com.mx/estacion/5227/armando-emmanuel-villa-dominguez/


In [10]:
# Los links ya son URLs completas, úsalos directo
url_detalle = links[0]  # ← sin concatenar nada
print(f"Scrapeando: {url_detalle}\n")

response2 = requests.get(url_detalle, headers=headers, timeout=15)
soup2     = BeautifulSoup(response2.text, "html.parser")

# ── Busca el Permiso ─────────────────────────────────────────
import re

permiso_tag = soup2.find("strong", string=re.compile("Permiso"))
if permiso_tag:
    permiso = permiso_tag.find_next_sibling(string=True)
    print(f"Permiso: {permiso.strip()}")

# ── Busca coordenadas en los scripts ─────────────────────────
scripts = soup2.find_all("script")
for i, script in enumerate(scripts):
    if script.string and re.search(r'-?\d{2,3}\.\d{4,}', script.string):
        print(f"\nScript #{i} con coordenadas:")
        print(script.string[:500])

Scrapeando: https://gasolinamexico.com.mx/estacion/13392/aceites-y-combustibles-jr-sa-de-cv/

Permiso: PL/11257/EXP/ES/2015

Script #7 con coordenadas:

	var map = L.map('map').setView([29.17184, -110.9152], 16);

    const mapLink = 
        '<a href="https://openstreetmap.org">OpenStreetMap</a>';
    L.tileLayer(
        'https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
        attribution: '&copy; ' + mapLink + ' Contributors',
        maxZoom: 18,
        }).addTo(map);

	const marker = new L.marker([29.17184,-110.9152])
				.bindPopup("ACEITES Y COMBUSTIBLES J.R. S.A. DE C.V.")
				.addTo(map);

	var popup = L.popup();

	map.scrollWh

Script #8 con coordenadas:

	{
	"@context": "http://schema.org",
	"@type": "LocalBusiness",
	"aggregateRating": {
	"@type": "AggregateRating",
	"ratingValue": "4.5",
	"reviewCount": "10"
	},
	"address": {
	"@type": "PostalAddress",
	"addressLocality": "Hermosillo",
	"addressRegion": "Sonora",
	"streetAddress": "Carretera Internacional a Nogales

In [ ]:
import requests
import re
from bs4 import BeautifulSoup

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def scrape_detalle_estacion(url: str) -> dict | None:
    """
    Dado el link de detalle de una estación, extrae:
    - Permiso  (llave de join con CRE API)
    - Latitud
    - Longitud
    """
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"❌ Error de red: {e}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # ── 1. Permiso ───────────────────────────────────────────────
    permiso = None
    permiso_tag = soup.find("strong", string=re.compile(r"Permiso"))
    if permiso_tag:
        raw = permiso_tag.find_next_sibling(string=True)
        if raw:
            permiso = raw.strip()

    # ── 2. Coordenadas del script Leaflet ────────────────────────
    lat, lng = None, None
    for script in soup.find_all("script"):
        if not script.string:
            continue
        # Busca: L.map('map').setView([29.17184, -110.9152], 16)
        coord_match = re.search(
            r'setView\($$(-?\d+\.\d+),\s*(-?\d+\.\d+)$$',
            script.string
        )
        if coord_match:
            lat = float(coord_match.group(1))
            lng = float(coord_match.group(2))
            break

    if not permiso or lat is None:
        print(f"⚠️  Datos incompletos en {url}")
        return None

    return {
        "permiso" : permiso,
        "latitud" : lat,
        "longitud": lng,
    }

# ── Prueba con los primeros 3 links ──────────────────────────────
for link in links[:3]:
    resultado = scrape_detalle_estacion(link)
    print(resultado)
    print()

In [ ]:
def scrape_ubicaciones_municipio(links: list[str], delay: float = 1.0) -> dict:
    """
    Recorre todos los links de un municipio y devuelve:
    { "PL/11257/EXP/ES/2015": {"latitud": 29.17, "longitud": -110.9} }
    """
    import time
    ubicaciones = {}

    for i, link in enumerate(links, 1):
        print(f"  [{i}/{len(links)}] {link.split('/')[-2]}")
        
        data = scrape_detalle_estacion(link)
        
        if data:
            ubicaciones[data["permiso"]] = {
                "latitud" : data["latitud"],
                "longitud": data["longitud"],
            }
        
        time.sleep(delay)

    print(f"\n✅ {len(ubicaciones)}/{len(links)} estaciones con coordenadas")
    return ubicaciones

In [ ]:
# df ya tiene columna "Numero" = "PL/11257/EXP/ES/2015"
df["Latitud"]  = df["Numero"].map(lambda x: ubicaciones.get(x, {}).get("latitud"))
df["Longitud"] = df["Numero"].map(lambda x: ubicaciones.get(x, {}).get("longitud"))